In [6]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, Bidirectional, GlobalAveragePooling1D,
    Conv1D, BatchNormalization, Activation, RepeatVector,
    TimeDistributed, Attention, Concatenate, AdditiveAttention,Attention)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import layers, Model, Input

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc, os, time
from keras import backend as K
from vmdpy import VMD

Yt = pd.read_csv('ws.csv', header=1, parse_dates=['Timestamp'])
Yt = Yt.rename(columns={'Timestamp': 'time'})

Yt['wind_sin'] = np.sin(np.deg2rad(Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360))
Yt['wind_cos'] = np.cos(np.deg2rad(Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360))

# 2. 湍流强度 SD（四层）轻度 clip
sd_cols = [
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s']

def clip_sd(col):
    Q1 = Yt[col].quantile(0.25)
    Q3 = Yt[col].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 3 * IQR
    Yt[col] = np.clip(Yt[col], None, upper)

for c in sd_cols:
    clip_sd(c)

# 3. 湍流强度 TI = SD / Mean
Yt['TI_110'] = Yt['Ch1_Anem_110.00m_E_SD_m/s'] / Yt['Ch1_Anem_110.00m_E_Avg_m/s']
Yt['TI_50']  = Yt['Ch2_Anem_50.00m_E_SD_m/s']  / Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['TI_30']  = Yt['Ch3_Anem_30.00m_E_SD_m/s']  / Yt['Ch3_Anem_30.00m_E_Avg_m/s']
Yt['TI_10']  = Yt['Ch4_Anem_10.00m_E_SD_m/s']  / Yt['Ch4_Anem_10.00m_E_Avg_m/s']

Yt.replace([np.inf, -np.inf], np.nan, inplace=True)
Yt.fillna(0, inplace=True)

# 4. 阵风偏差（gust deviation）
Yt['gust_dev_110'] = Yt['Ch1_Anem_110.00m_E_Gust_m/s'] - Yt['Ch1_Anem_110.00m_E_Avg_m/s']
Yt['gust_dev_50']  = Yt['Ch2_Anem_50.00m_E_Gust_m/s']  - Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['gust_dev_30']  = Yt['Ch3_Anem_30.00m_E_Gust_m/s']  - Yt['Ch3_Anem_30.00m_E_Avg_m/s']
Yt['gust_dev_10']  = Yt['Ch4_Anem_10.00m_E_Gust_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']

# 5. 层间风切变（vertical shear）
Yt['shear_110_10'] = Yt['Ch1_Anem_110.00m_E_Avg_m/s'] - Yt['Ch4_Anem_10.00m_E_Avg_m/s']
Yt['shear_110_50'] = Yt['Ch1_Anem_110.00m_E_Avg_m/s'] - Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['shear_50_10']  = Yt['Ch2_Anem_50.00m_E_Avg_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']
Yt['shear_30_10']  = Yt['Ch3_Anem_30.00m_E_Avg_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']

# 6. 气象变量：温度 + 气压
Yt['temp'] = Yt['Ch9_Analog_10.00m_N_Avg_C']
Yt['pressure'] = Yt['Ch10_Analog_10.00m_N_Avg_kpa']

# 7. 时间特征（分钟周期）
Yt['minute'] = Yt['time'].dt.minute
Yt['minute_sin'] = np.sin(2 * np.pi * Yt['minute'] / 60)
Yt['minute_cos'] = np.cos(2 * np.pi * Yt['minute'] / 60)

features = [

    # 原始风速
    'Ch4_Anem_10.00m_E_Avg_m/s',
    'Ch3_Anem_30.00m_E_Avg_m/s',
    'Ch2_Anem_50.00m_E_Avg_m/s',
    'Ch1_Anem_110.00m_E_Avg_m/s',

    # 风向
    'wind_sin', 'wind_cos',

    # SD turbulence
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s',

    # TI
    'TI_110', 'TI_50', 'TI_30', 'TI_10',

    # gust deviation
    'gust_dev_110', 'gust_dev_50', 'gust_dev_30', 'gust_dev_10',

    # shear
    'shear_110_10', 'shear_110_50', 'shear_50_10', 'shear_30_10',

    # 气象变量
    'temp', 'pressure',

    # 时间周期
    'minute_sin', 'minute_cos']

targets = [
    'Ch1_Anem_110.00m_E_Avg_m/s',
    'Ch2_Anem_50.00m_E_Avg_m/s',
    'Ch3_Anem_30.00m_E_Avg_m/s',
    'Ch4_Anem_10.00m_E_Avg_m/s']
target_h = ['Ch2_Anem_50.00m_E_Avg_m/s']

In [ ]:

def dataset(data_scaled, window_size, pred_steps, target_idx):
    X, y = [], []
    n = len(data_scaled)
    for i in range(n - window_size - pred_steps):
        X.append(data_scaled[i:i + window_size, :])
        y.append(data_scaled[i + window_size:i + window_size + pred_steps, target_idx])
    return np.array(X), np.array(y)


def mape(true, pred):
    true = np.asarray(true)
    pred = np.asarray(pred)
    return np.mean(np.abs((true - pred) / np.maximum(np.abs(true), 1e-6))) * 100


def make_y_seq_from_real(y_real_1d, window_size, pred_steps):
    ys = []
    n = len(y_real_1d)
    for i in range(n - window_size - pred_steps):
        ys.append(y_real_1d[i + window_size:i + window_size + pred_steps, 0])
    return np.array(ys)

w_list = [3,6, 12, 24]
f_list = [1, 2, 3, 4, 5, 6]
base_dir = "4-50m"
os.makedirs(base_dir, exist_ok=True)

results = []
predictions = {}

np.random.seed(42)
tf.random.set_seed(42)

for w in w_list:
    for f in f_list:
        forecast_min = f * 10
        print(f"\nTraining w={w} | horizon={forecast_min} min")

        for target in target_h:
            df_y = Yt[['time'] + features].copy()
            df_y = df_y.sort_values('time').reset_index(drop=True)

            split_idx = int(0.8 * len(df_y))
            train_df = df_y[features].iloc[:split_idx]
            test_df  = df_y[features].iloc[split_idx:]

            y_train_real = train_df[[target]].values.astype(float)
            y_test_real  = test_df[[target]].values.astype(float)

            scaler_X = MinMaxScaler()
            train_scaled = scaler_X.fit_transform(train_df.values.astype(float))
            test_scaled  = scaler_X.transform(test_df.values.astype(float))

            target_idx = features.index(target)

            X_train, _ = dataset(train_scaled, w, f, target_idx)
            X_test,  _ = dataset(test_scaled,  w, f, target_idx)

            y_train_seq_real = make_y_seq_from_real(y_train_real, w, f)
            y_test_seq_real  = make_y_seq_from_real(y_test_real,  w, f)

            scaler_y = MinMaxScaler()
            scaler_y.fit(y_train_real)

            y_train_s = scaler_y.transform(y_train_seq_real.reshape(-1, 1)).reshape(-1, f, 1)
            y_test_s  = scaler_y.transform(y_test_seq_real.reshape(-1, 1)).reshape(-1, f, 1)

            encoder_inputs = Input(shape=(w, X_train.shape[2]))
            _, state_h, state_c = LSTM(
                128, activation="tanh",
                return_state=True
            )(encoder_inputs)

            decoder_inputs = layers.Lambda(
                lambda x: tf.zeros((tf.shape(x)[0], f, 1))
            )(encoder_inputs)

            decoder_outputs = LSTM(
                128, activation="tanh",
                return_sequences=True
            )(decoder_inputs, initial_state=[state_h, state_c])

            outputs = TimeDistributed(Dense(1))(decoder_outputs)

            model = Model(encoder_inputs, outputs)
            model.compile(optimizer="adam", loss="mse")

            start = time.time()
            model.fit(
                X_train, y_train_s,
                epochs=150,
                batch_size=128,
                validation_split=0.1,
                shuffle=False,
                callbacks=[
                    ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-5),
                    EarlyStopping(patience=10, restore_best_weights=True)
                ],
                verbose=0
            )
            elapsed = time.time() - start

            y_train_pred_s = model.predict(X_train, verbose=0)
            y_test_pred_s  = model.predict(X_test,  verbose=0)

            y_train_pred_last = scaler_y.inverse_transform(
                y_train_pred_s[:, -1, :].reshape(-1, 1)
            ).flatten()

            y_test_pred_last = scaler_y.inverse_transform(
                y_test_pred_s[:, -1, :].reshape(-1, 1)
            ).flatten()

            y_train_true_last = y_train_seq_real[:, -1]
            y_test_true_last  = y_test_seq_real[:, -1]

            rmse_train = np.sqrt(np.mean((y_train_true_last - y_train_pred_last) ** 2))
            rmse_test  = np.sqrt(np.mean((y_test_true_last  - y_test_pred_last ) ** 2))

            r2_train = r2_score(y_train_true_last, y_train_pred_last)
            r2_test  = r2_score(y_test_true_last,  y_test_pred_last)

            mae_train = mean_absolute_error(y_train_true_last, y_train_pred_last)
            mae_test  = mean_absolute_error(y_test_true_last,  y_test_pred_last)

            mape_train = mape(y_train_true_last, y_train_pred_last)
            mape_test  = mape(y_test_true_last,  y_test_pred_last)

            print(
                f"Test RMSE={rmse_test:.4f}, "
                f"R2={r2_test:.4f}, "
                f"MAE={mae_test:.4f}, "
                f"MAPE={mape_test:.2f}%"
            )

            results.append({
                "Height": target,
                "Forecast_Min": forecast_min,
                "Window": w,
                "Test_RMSE": rmse_test,
                "Test_R2": r2_test,
                "Test_MAE": mae_test,
                "Test_MAPE": mape_test,
                "Elapsed_Seconds": elapsed
            })

            predictions[(w, f, target)] = {
                "test_true_last": y_test_true_last,
                "test_pred_last": y_test_pred_last
            }

            K.clear_session()
            gc.collect()

df_results = pd.DataFrame(results)

for w in w_list:
    w_dir = os.path.join(base_dir, f"w{w}")
    os.makedirs(w_dir, exist_ok=True)

    df_w = df_results[df_results["Window"] == w]
    df_w.to_csv(os.path.join(w_dir, f"summary_w{w}.csv"), index=False)



Training w=3 | horizon=10 min
Test RMSE=0.8477, R2=0.9577, MAE=0.6231, MAPE=28.97%

Training w=3 | horizon=20 min
Test RMSE=1.2247, R2=0.9118, MAE=0.8950, MAPE=49.40%

Training w=3 | horizon=30 min
Test RMSE=1.4751, R2=0.8720, MAE=1.0835, MAPE=67.41%

Training w=3 | horizon=40 min
Test RMSE=1.6636, R2=0.8371, MAE=1.2266, MAPE=78.40%

Training w=3 | horizon=50 min
Test RMSE=1.8210, R2=0.8048, MAE=1.3519, MAPE=95.16%

Training w=3 | horizon=60 min
Test RMSE=1.9649, R2=0.7727, MAE=1.4722, MAPE=104.38%

Training w=6 | horizon=10 min
Test RMSE=0.8425, R2=0.9582, MAE=0.6206, MAPE=29.53%

Training w=6 | horizon=20 min
Test RMSE=1.2246, R2=0.9117, MAE=0.8961, MAPE=49.38%

Training w=6 | horizon=30 min
Test RMSE=1.4692, R2=0.8729, MAE=1.0833, MAPE=73.95%

Training w=6 | horizon=40 min
Test RMSE=1.6568, R2=0.8384, MAE=1.2244, MAPE=79.80%

Training w=6 | horizon=50 min
Test RMSE=1.8180, R2=0.8054, MAE=1.3526, MAPE=98.13%

Training w=6 | horizon=60 min
Test RMSE=1.9487, R2=0.7763, MAE=1.4577, MAP

In [ ]:
import os
import re
import pandas as pd

base_dir = "4-110m"
os.makedirs(base_dir, exist_ok=True)

w_dir = os.path.join(base_dir, f"w{w}")
os.makedirs(w_dir, exist_ok=True)

safe_h = re.sub(r'[^A-Za-z0-9]+', "_", target_h[0])

for (w_i, f_i, target_i), data in predictions.items():

    w_dir = os.path.join(base_dir, f"w{w_i}")
    os.makedirs(w_dir, exist_ok=True)

    safe_h = re.sub(r'[^A-Za-z0-9]+', "_", target_i)

    df_test = pd.DataFrame({
        "True_Test": data["test_true_real"],
        "Pred_Test": data["test_pred_real"]
    })

    filename = f"{safe_h}_test_{f_i*10}min.csv"
    df_test.to_csv(os.path.join(w_dir, filename), index=False)


df_results = pd.DataFrame(results)

for w_i in w_list:
    w_dir = os.path.join(base_dir, f"w{w_i}")
    os.makedirs(w_dir, exist_ok=True)

    df_w = df_results[df_results["Forecast_Min"].notna()]
    df_w = df_w[df_w["Height"] == target_h[0]]

    summary_path = os.path.join(w_dir, f"summary_w{w_i}.csv")
    df_w.to_csv(summary_path, index=False)

colors = ["#006699", "#b30000", "#009933",
          "#ff9900", "#660066", "#666600"]

plt.rcParams["font.size"] = 13

def plot_saved_by_w(predictions_dict, w_i):

    w_dir = os.path.join(base_dir, f"w{w_i}")
    os.makedirs(w_dir, exist_ok=True)

    idx = 0
    for (w_k, f_k, target_k), data in predictions_dict.items():

        if w_k != w_i:
            continue

        true_vals = data["test_true_real"]
        pred_vals = data["test_pred_real"]

        rmse = np.sqrt(np.mean((true_vals - pred_vals) ** 2))
        r2 = r2_score(true_vals, pred_vals)

        forecast_min = f_k * 10

        min_val = min(true_vals.min(), pred_vals.min())
        max_val = max(true_vals.max(), pred_vals.max())

        plt.figure(figsize=(7, 7))
        plt.scatter(
            true_vals,
            pred_vals,
            alpha=0.35,
            color=colors[idx % len(colors)],
            edgecolor="none"
        )

        plt.plot([min_val, max_val], [min_val, max_val], "r--", lw=2)

        plt.xlabel("True Wind Speed (m/s)")
        plt.ylabel("Predicted Wind Speed (m/s)")
        plt.title(f"{forecast_min}-Minute Ahead Prediction (w={w_i})")

        plt.text(
            min_val,
            max_val,
            f"$R^2={r2:.4f}$\nRMSE={rmse:.4f}$",
            va="top",
            bbox=dict(facecolor="white", alpha=0.85)
        )

        plt.grid(alpha=0.35)
        plt.tight_layout()
        plt.savefig(os.path.join(w_dir, f"scatter_{forecast_min}min.png"), dpi=300)
        plt.close()

        idx += 1

for w_i in w_list:
    plot_saved_by_w(predictions, w_i)